# ConceptNet-100k: `RelatedTo` grammar ablation — old + today + new L4 runs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SemanticMap/semgraphex/blob/feature/recursive-lossless-graph-dictionary-v2/notebooks/17_relatedto_grammar_ablation_cn100k_l4_colab.ipynb)

Проверяются две разные гипотезы и поэтому два режима нельзя смешивать:

1. **Retrospective codec ablation** — берём уже найденные accepted occurrences из старого run `-01` и сегодняшнего `-03`, не повторяем Wishart/MDL, но пере-кодируем их так, чтобы grammar-internal relation был только `RelatedTo`; все остальные relation остаются exact residual. Это измеряет чистый эффект relation-фрагментации кодека.
2. **Prospective discovery ablation** — запускаем новые hierarchy runs: (A) граф только из `RelatedTo`; (B) полный граф, но discovery/identity/grammar используют только `RelatedTo`, а остальные relation сохраняются lossless в residual.

Основные метрики: `compression_ratio_binary_baseline`, `residual_share`, `reuse_rate`, размер словаря, accepted occurrences, recursive types и spectral gap. `ratio < 1` означает выигрыш над compact exact binary baseline.

In [ ]:
# ===== PARAMETERS =====
REPO_URL = 'https://github.com/SemanticMap/semgraphex.git'
BRANCH = 'feature/recursive-lossless-graph-dictionary-v2'
PINNED_SHA = '639f4c8cda219a3e9e353cbcb23cbb32ff6ad498'
REPO_ROOT = '/content/semgraphex'
DRIVE_ROOT = '/content/drive/MyDrive/SemanticMap/colab/wishart'
DATASET_DRIVE = 'data/conceptnet_en_100k.tsv'

HISTORICAL_RUNS = [
    'wishart-grammar-v2-cn100k-l4-01',
    'wishart-grammar-v2-cn100k-l4-03',
]
RUN_RETROSPECTIVE = True
RUN_PROSPECTIVE = True

PROSPECTIVE = [
    {
        'label': 'relatedto_only_graph',
        'config': 'configs/wishart_conceptnet_grammar_v2_relatedto_only.yaml',
        'run_name': 'wishart-grammar-v2-cn100k-relatedto-only-l4-01',
    },
    {
        'label': 'relatedto_grammar_fullgraph',
        'config': 'configs/wishart_conceptnet_grammar_v2_relatedto_grammar_fullgraph.yaml',
        'run_name': 'wishart-grammar-v2-cn100k-relatedto-grammar-fullgraph-l4-01',
    },
]

CPU_WORKERS = 12
BLAS_THREADS = 1
DEVICE = 'cuda'
REQUIRE_L4 = True
SCRATCH = '/content/semmap-relatedto-ablation'
EXPERIMENT_DRIVE = 'experiments/relatedto_ablation_01'

assert BLAS_THREADS == 1


In [ ]:
# ===== RUNTIME PREFLIGHT =====
import json, os, shutil, subprocess, sys, time
from pathlib import Path

print('Python:', sys.version.replace('\n',' '))
print('CPU count:', os.cpu_count())
disk = shutil.disk_usage('/content')
print('Local free GiB:', round(disk.free / 2**30, 2))
if not shutil.which('nvidia-smi'):
    raise RuntimeError('GPU not found; choose Runtime → Change runtime type → L4 GPU')
gpu = subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total,memory.free','--format=csv,noheader'],text=True).strip()
print('GPU:',gpu)
if REQUIRE_L4 and 'L4' not in gpu.upper():
    raise RuntimeError(f'Expected L4, got: {gpu}')


In [ ]:
# ===== MOUNT DRIVE =====
from google.colab import drive
drive.mount('/content/drive')
DRIVE = Path(DRIVE_ROOT)
DATASET = DRIVE / DATASET_DRIVE
if not DATASET.is_file():
    raise FileNotFoundError(DATASET)
EXPERIMENT = DRIVE / EXPERIMENT_DRIVE
EXPERIMENT.mkdir(parents=True, exist_ok=True)
Path(SCRATCH).mkdir(parents=True, exist_ok=True)
print('Dataset:', DATASET)
print('Experiment artifacts:', EXPERIMENT)


In [ ]:
# ===== PINNED CHECKOUT + INSTALL =====
repo = Path(REPO_ROOT)
if not (repo/'.git').exists():
    subprocess.run(['git','clone','--branch',BRANCH,'--single-branch',REPO_URL,str(repo)],check=True)
else:
    subprocess.run(['git','-C',str(repo),'fetch','origin',BRANCH],check=True)
subprocess.run(['git','-C',str(repo),'checkout','--detach',PINNED_SHA],check=True)
sha = subprocess.check_output(['git','-C',str(repo),'rev-parse','HEAD'],text=True).strip()
assert sha == PINNED_SHA, (sha, PINNED_SHA)
constraints = repo/'requirements/constraints-colab.txt'
subprocess.run([sys.executable,'-m','pip','install','-q','-c',str(constraints),'-e',f'{repo}[dev,wishart,gpu]'],check=True)
import numpy as np, pandas as pd
from scipy import sparse
from semmap_haken.relation_ablation import baseline_run_rows, retrospective_related_relation
print('Pinned SHA:',sha)


In [ ]:
# ===== STAGE ONLY NEEDED HISTORICAL ARTIFACTS TO /content =====
def stage_run_minimal(run_name: str) -> Path:
    source = DRIVE/'runs'/run_name
    if not source.is_dir():
        raise FileNotFoundError(source)
    target = Path(SCRATCH)/'history'/run_name
    target.mkdir(parents=True, exist_ok=True)
    for transition in sorted(source.glob('transition_*_*')):
        parts = transition.name.split('_')
        source_level = int(parts[1])
        level_source = source/f'level_{source_level:03d}'
        level_target = target/f'level_{source_level:03d}'
        (level_target/'relations').mkdir(parents=True, exist_ok=True)
        for name in ['adjacency.npz','symbolic_nodes.jsonl','dynamic_metrics.json']:
            src = level_source/name
            if src.is_file() and not (level_target/name).is_file():
                shutil.copy2(src, level_target/name)
        for src in (level_source/'relations').glob('*'):
            dst = level_target/'relations'/src.name
            if src.is_file() and not dst.is_file(): shutil.copy2(src,dst)
        trans_target = target/transition.name
        trans_target.mkdir(exist_ok=True)
        for name in ['figure_occurrences.jsonl','dictionary_metrics.json','wishart.json']:
            src = transition/name
            if src.is_file() and not (trans_target/name).is_file():
                shutil.copy2(src, trans_target/name)
    return target

staged_runs = {name: stage_run_minimal(name) for name in HISTORICAL_RUNS}
print({k:str(v) for k,v in staged_runs.items()})


In [ ]:
# ===== BASELINE METRICS FROM OLD + TODAY RUNS =====
baseline_rows = []
for run_name, run_path in staged_runs.items():
    baseline_rows.extend(baseline_run_rows(run_path))
baseline_df = pd.DataFrame(baseline_rows)
cols = [c for c in [
    'run','source_level','node_count','spectral_gap','wishart_clusters',
    'canonical_types','accepted_occurrences','dictionary_size','reuse_rate',
    'recursive_types_total','compression_ratio_binary','residual_share'
] if c in baseline_df.columns]
display(baseline_df[cols].sort_values(['run','source_level']))


In [ ]:
# ===== RETROSPECTIVE: SAME FIGURES, RelatedTo GRAMMAR ONLY =====
# Это codec-only ablation: Wishart/MDL/occurrences не пересчитываются.
retro_rows = []
if RUN_RETROSPECTIVE:
    persistent_reports = EXPERIMENT/'retrospective'
    persistent_reports.mkdir(parents=True, exist_ok=True)
    for run_name, staged in staged_runs.items():
        print('\nRETROSPECTIVE:',run_name)
        local_reports = Path(SCRATCH)/'retro_reports'/run_name
        # Restore already completed reports from Drive so rerun is resumable.
        drive_run_reports = persistent_reports/run_name
        drive_run_reports.mkdir(parents=True, exist_ok=True)
        local_reports.mkdir(parents=True, exist_ok=True)
        for src in drive_run_reports.glob('*.json'):
            shutil.copy2(src, local_reports/src.name)
        rows = retrospective_related_relation(
            staged,
            grammar_relations=('RelatedTo',),
            report_dir=local_reports,
        )
        retro_rows.extend(rows)
        for src in local_reports.glob('*.json'):
            shutil.copy2(src, drive_run_reports/src.name)
retro_df = pd.DataFrame(retro_rows)
if len(retro_df):
    display(retro_df[[
        'run','source_level','accepted_occurrences','compression_ratio_binary',
        'residual_share','internal_edge_records','port_edge_records',
        'residual_edge_records','non_grammar_relation_edge_records','roundtrip_exact'
    ]].sort_values(['run','source_level']))


In [ ]:
# ===== DIRECT CODEC EFFECT: BASELINE vs RETROSPECTIVE =====
if len(retro_df):
    left = baseline_df[['run','source_level','compression_ratio_binary','residual_share']].rename(columns={
        'compression_ratio_binary':'baseline_ratio',
        'residual_share':'baseline_residual_share',
    })
    right = retro_df[['run','source_level','compression_ratio_binary','residual_share']].rename(columns={
        'compression_ratio_binary':'relatedto_ratio_same_figures',
        'residual_share':'relatedto_residual_share_same_figures',
    })
    codec_compare = left.merge(right,on=['run','source_level'],how='inner')
    codec_compare['ratio_delta'] = codec_compare['relatedto_ratio_same_figures'] - codec_compare['baseline_ratio']
    codec_compare['residual_share_delta'] = codec_compare['relatedto_residual_share_same_figures'] - codec_compare['baseline_residual_share']
    codec_compare['relatedto_codec_better'] = codec_compare['ratio_delta'] < 0
    display(codec_compare.sort_values(['run','source_level']))


In [ ]:
# ===== PROSPECTIVE NEW/RESUME HELPER =====
def run_ablation(spec):
    run_name = spec['run_name']
    run_dir = DRIVE/'runs'/run_name
    config = repo/spec['config']
    completed = run_dir/'COMPLETED'
    resume = False
    if completed.is_file():
        print('SKIP COMPLETED:',run_name)
        return run_dir
    if (run_dir/'input.json').is_file():
        manifest = json.loads((run_dir/'input.json').read_text())
        if manifest.get('code_revision') != PINNED_SHA:
            raise RuntimeError(
                f'Cannot resume {run_name}: checkpoint revision={manifest.get("code_revision")} '
                f'but notebook revision={PINNED_SHA}'
            )
        if int((manifest.get('execution_options') or {}).get('cpu_workers',-1)) != CPU_WORKERS:
            raise RuntimeError(f'CPU_WORKERS mismatch for {run_name}')
        if int(manifest.get('blas_threads',-1)) != BLAS_THREADS:
            raise RuntimeError(f'BLAS_THREADS mismatch for {run_name}')
        resume = True
    cmd = [
        sys.executable,'-m','semmap_haken.wishart_colab_cli',
        '--config',str(config),'--drive-root',DRIVE_ROOT,
        '--scratch-root',str(Path(SCRATCH)/'prospective'),
        '--run-name',run_name,'--device',DEVICE,
        '--cpu-workers',str(CPU_WORKERS),'--blas-threads',str(BLAS_THREADS),
        '--compare-mode','size_mtime','--dataset-drive',DATASET_DRIVE,
    ]
    if resume: cmd.append('--resume')
    print(('RESUME' if resume else 'NEW'),run_name)
    subprocess.run(cmd,cwd=repo,check=True)
    return run_dir


In [ ]:
# ===== RUN THE TWO CLEAN ABLATIONS =====
prospective_runs = {}
if RUN_PROSPECTIVE:
    for spec in PROSPECTIVE:
        prospective_runs[spec['label']] = run_ablation(spec)
prospective_runs


In [ ]:
# ===== FINAL COMPARISON TABLE =====
final_rows = list(baseline_rows)
for label, run_dir in prospective_runs.items():
    for row in baseline_run_rows(run_dir):
        row = dict(row)
        row['mode'] = label
        final_rows.append(row)
final_df = pd.DataFrame(final_rows)
show = [c for c in [
    'mode','run','source_level','node_count','spectral_gap','wishart_clusters',
    'canonical_types','accepted_occurrences','dictionary_size','reuse_rate',
    'recursive_types_total','compression_ratio_binary','residual_share',
    'internal_edge_records','port_edge_records','residual_edge_records'
] if c in final_df.columns]
display(final_df[show].sort_values(['mode','run','source_level']))

out = EXPERIMENT/'comparison_rows.json'
out.write_text(json.dumps(final_rows,indent=2,sort_keys=True,default=str)+'\n')
print('Saved:',out)

# Scientific checks, not a binary verdict on the whole research program.
if len(retro_df):
    improved = int((codec_compare['ratio_delta'] < 0).sum())
    print(f'Retrospective codec improvement: {improved}/{len(codec_compare)} transitions')
print('Primary prospective comparison: relatedto_grammar_fullgraph vs baseline_multirelation at comparable levels.')
print('RelatedTo-only graph is an upper-bound/topology ablation; its node universe may differ and raw bytes are not directly comparable.')


## Как интерпретировать результат

- **Retrospective ratio уменьшается** при тех же occurrences → relation labels действительно фрагментируют exact shape/interface coding.
- **Prospective `relatedto_grammar_fullgraph` повышает reuse и уменьшает residual/compression ratio** → `RelatedTo` полезен не только для кодека, но и как более устойчивый discovery-space; остальные relation можно рассматривать как correction layer.
- **Только `relatedto_only_graph` улучшается**, а fullgraph-residual нет → выигрыш объясняется главным образом удалением специализированных relation, а не лучшей грамматикой полного графа.
- Сравнивайте физический `compression_ratio_binary_baseline`, а не только node reduction. Сохранение spectral gap анализируется отдельно от storage compression.

Все новые hierarchy runs имеют собственные `RUN_NAME`; старые `-01` и сегодняшние `-03` никогда не продолжаются новым кодом и используются только как read-only baseline.